# Scene Search POC

**Question this notebook answers:** can we take a film, and find *"the scene where the man ties the other man upside down"* — by meaning, not by keyword?

Nothing else in the movie-streamer project matters if the answer is no. So we answer it here first, on one clip, with no services, no database and no Docker.

### Run this on Kaggle
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. Run all cells top to bottom.

The GPU is deliberate: we are testing whether the *idea* works, not whether a laptop can run it. What fits on CPU is a later problem with a known fix.

### The pipeline

```
clip ──▶ shot detection ──▶ keyframes ──┬──▶ CLIP visual embedding
                                        └──▶ VLM caption ──┐
         audio ──▶ Whisper transcript ───────────────────┬─┴──▶ text embedding
                                                         │
                        shots ──▶ merged into scenes ◀───┘
                                        │
                    query ──▶ 3 rankings ──▶ RRF fusion ──▶ scene + start_ms
```

**shot** = one camera take (~2-5s). We embed these — small and precise.
**scene** = one dramatic unit, many shots. We return these — enough context to be useful, and the correct place to seek.

That parent-child split is the core design decision. See `docs/concepts/01-chunking-in-rag.md`.

---
## 0. Setup

In [ ]:
!pip install -q --upgrade pip
!pip install -q "scenedetect[opencv]" faster-whisper sentence-transformers accelerate qwen-vl-utils
!pip install -q --upgrade transformers
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

import importlib, shutil

REQUIRED = {
    "scenedetect": "scenedetect[opencv]",
    "faster_whisper": "faster-whisper",
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers",
    "cv2": "opencv-python",
    "torch": "torch",
}

missing = []
for mod, pkg in REQUIRED.items():
    try:
        m = importlib.import_module(mod)
        print(f"  ok    {mod:22} {getattr(m, '__version__', '')}")
    except Exception as e:
        missing.append(pkg)
        print(f"  FAIL  {mod:22} {type(e).__name__}: {e}")

if not shutil.which("ffmpeg"):
    missing.append("ffmpeg")
    print("  FAIL  ffmpeg                 not on PATH")

if missing:
    raise RuntimeError(
        f"missing: {missing}\n"
        "On Kaggle this almost always means the pip install could not reach the network.\n"
        "Right panel -> Settings -> Internet: On (needs a phone-verified account), then re-run this cell."
    )

print("\nall dependencies present")

In [ ]:
import os, re, json, time, math, subprocess, urllib.request, textwrap
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(exist_ok=True, parents=True)
FRAMES = WORK / "frames"; FRAMES.mkdir(exist_ok=True)

print(f"device={DEVICE}  dtype={DTYPE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

### Knobs

Every number here is a tuning knob. Guessing them is how RAG systems end up mediocre — the labelled query set in section 8 is what turns each one from an argument into a measurement.

The two that matter most are `SHOT_THRESHOLD` (too high shatters scenes, too low merges unrelated action) and `SCENE_MERGE_SIM` (the main quality knob in the whole feature).

In [ ]:
CLIP_START_SEC   = 0
CLIP_DURATION_SEC = 300
INDEX_HEIGHT     = 480
SHOT_THRESHOLD   = 27.0
MIN_SHOT_SEC     = 0.6
SCENE_MERGE_SIM  = 0.75
SCENE_WINDOW     = 3
SCENE_MAX_SEC    = 25.0
SCENE_MIN_SEC    = 3.0
SCENE_MAX_GAP    = 2.0
CLIP_MODEL       = "openai/clip-vit-large-patch14"
WHISPER_MODEL    = "small"
VLM_MODEL        = "Qwen/Qwen2-VL-2B-Instruct"
TEXT_MODEL       = "BAAI/bge-base-en-v1.5"
RRF_K            = 10
FUSE_DEPTH       = 10
SIGNAL_Z         = 1.5
TOP_K            = 10
HIT_TOLERANCE_SEC = 5.0
TIMINGS = {}

---

## 1. Get a clip

Source for this run is the **Iron Man jet-chase clip** (5 min, 720p30). It is a deliberately
hard and deliberately *fair* test:

- **Action-dense, dialogue-sparse.** Long stretches are pure visual. If retrieval works here it
  is not quietly leaning on the transcript — which is the failure mode that makes a demo look
  smart right up until someone queries a scene where nobody speaks.
- **Fast cutting.** Shot detection gets stressed the way a real action film stresses it, not the
  way a slow clip does.
- **Repeated visual vocabulary.** Sky, jets and armour recur constantly, so near-duplicate shots
  have to be told apart by *what is happening*, not by *what is on screen*. That is exactly the
  gap the caption pass exists to close, and the ablation will show whether it actually does.

**On the content:** measuring retrieval quality against a file on your own disk is private
evaluation. What is not fine is this file reaching the repo, the README, a screenshot, or
anything shown to an interviewer — that is distribution. It is gitignored. The shipped demo runs
on CC-BY Blender open movies, so `SOURCE_URLS` below stays wired up for that.

**To run on Kaggle:** upload the file as a **private** Kaggle Dataset. The loader globs
`/kaggle/input/` for any video, so the dataset slug does not matter.

In [ ]:
SOURCE_URLS = [
    "https://download.blender.org/demo/movies/ToS/ToS-4k-1920.mov",
    "https://archive.org/download/tears-of-steel/tears_of_steel_1080p.mp4",
    "https://download.blender.org/durian/movies/Sintel.2010.1080p.mkv",
]

VIDEO_EXTS = (".mp4", ".mkv", ".mov", ".webm", ".avi")

def find_local():
    roots = [Path("/kaggle/input"), Path("."), Path("/home/siddharth/Desktop/movie_streamer")]
    for root in roots:
        if not root.exists():
            continue
        for f in sorted(root.rglob("*")):
            if f.suffix.lower() in VIDEO_EXTS and f.is_file() and f.stat().st_size > 1_000_000:
                return f
    return None

LOCAL_SOURCE = None
RAW = WORK / "source.mov"
CLIP_PATH = WORK / "clip.mp4"
AUDIO_PATH = WORK / "clip.wav"

def fetch_source():
    if LOCAL_SOURCE:
        return Path(LOCAL_SOURCE)
    found = find_local()
    if found:
        print(f"using local source: {found}")
        return found
    if RAW.exists() and RAW.stat().st_size > 1_000_000:
        return RAW
    for url in SOURCE_URLS:
        try:
            print(f"trying {url}")
            urllib.request.urlretrieve(url, RAW)
            if RAW.stat().st_size > 1_000_000:
                print(f"got {RAW.stat().st_size/1e6:.0f} MB")
                return RAW
        except Exception as e:
            print(f"  failed: {e}")
    raise RuntimeError("no source available - attach a Kaggle dataset or set LOCAL_SOURCE")

src = fetch_source()

Now trim to a workable length and downscale to 480p.

The downscale is not laziness — it is the same decision as ADR-010 in the master doc: **index the low-resolution rendition, never the mezzanine.** Decoding 1080p just to sample frames burns the exact resource that is the bottleneck, and CLIP resizes to 224px anyway. The information we need survives; the cost does not.

In [ ]:
src = Path(src)
if not src.exists():
    raise FileNotFoundError(f"source not found: {src}")
print(f"source: {src}  ({src.stat().st_size/1e6:.0f} MB)")

def run_ffmpeg(args, label):
    r = subprocess.run(args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{label} failed (exit {r.returncode}):\n{r.stderr[-2000:]}")
    return r

t0 = time.time()
run_ffmpeg([
    "ffmpeg", "-y", "-loglevel", "error",
    "-ss", str(CLIP_START_SEC), "-t", str(CLIP_DURATION_SEC),
    "-i", str(src),
    "-vf", f"scale=-2:{INDEX_HEIGHT}",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
    "-c:a", "aac", "-ac", "1",
    str(CLIP_PATH)
], "video transcode")

run_ffmpeg([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", str(CLIP_PATH), "-vn", "-ac", "1", "-ar", "16000",
    str(AUDIO_PATH)
], "audio extract")

TIMINGS["prepare_clip"] = time.time() - t0

for f in (CLIP_PATH, AUDIO_PATH):
    if not f.exists() or f.stat().st_size < 10_000:
        raise RuntimeError(f"ffmpeg reported success but {f} is missing or empty")

probe = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",str(CLIP_PATH)],
                       capture_output=True, text=True)
CLIP_SECONDS = float(probe.stdout.strip())
print(f"clip: {CLIP_PATH}")
print(f"      {CLIP_SECONDS:.1f}s  ({CLIP_PATH.stat().st_size/1e6:.0f} MB)  in {TIMINGS['prepare_clip']:.0f}s")

---
## 2. Shot detection — structural chunking

This is the first and most important chunking decision, and it is the direct analogue of splitting a document on paragraph boundaries rather than every 512 tokens.

A **cut** is video's paragraph break. It is *authored* — a human editor decided that one visual idea ended and another began. Sampling a frame every N seconds instead would land mid-motion, mid-transition, on blur that means nothing.

It is also what makes the whole thing computationally tractable: a 10-minute clip has ~15,000 frames but only a few hundred shots. That reduction is roughly 150x, and it is the reason CPU-only inference is even conceivable later.

In [ ]:
from scenedetect import detect, ContentDetector

if not CLIP_PATH.exists():
    raise FileNotFoundError(
        f"{CLIP_PATH} does not exist.\n"
        "The ffmpeg cell in section 1 has not run successfully in this kernel.\n"
        "Use Run -> Run All: this notebook is a strict chain and cells cannot be run out of order."
    )

t0 = time.time()
raw_shots = detect(str(CLIP_PATH), ContentDetector(threshold=SHOT_THRESHOLD))
TIMINGS["shot_detection"] = time.time() - t0

@dataclass
class Shot:
    idx: int
    start: float
    end: float
    frame_path: str = ""
    caption: str = ""
    dialogue: str = ""
    scene_id: int = -1
    @property
    def mid(self): return (self.start + self.end) / 2
    @property
    def duration(self): return self.end - self.start

shots = []
for a, b in raw_shots:
    s, e = a.get_seconds(), b.get_seconds()
    if e - s < MIN_SHOT_SEC and shots:
        shots[-1].end = e
        continue
    shots.append(Shot(idx=len(shots), start=s, end=e))

if not shots:
    print(f"no cuts detected at threshold {SHOT_THRESHOLD} - falling back to fixed 5s windows")
    t = 0.0
    while t < CLIP_SECONDS:
        shots.append(Shot(idx=len(shots), start=t, end=min(t + 5.0, CLIP_SECONDS)))
        t += 5.0

for i, s in enumerate(shots):
    s.idx = i

mean_len = float(np.mean([s.duration for s in shots]))
print(f"{len(shots)} shots in {TIMINGS['shot_detection']:.0f}s")
print(f"mean shot length {mean_len:.1f}s")
print(f"compression: {int(CLIP_SECONDS*24)} frames -> {len(shots)} shots  ({int(CLIP_SECONDS*24/len(shots))}x)")
if mean_len < 1.5:
    print(f"\nmean is short for SHOT_THRESHOLD={SHOT_THRESHOLD} - raise it to merge more aggressively")

### Keyframes

One representative frame per shot, taken from the middle rather than the start — the first frame of a shot is often still mid-transition or mid-fade.

In [ ]:
if not shots:
    raise RuntimeError("no shots defined - run the shot detection cell above first")

probe_cap = cv2.VideoCapture(str(CLIP_PATH))
FPS = probe_cap.get(cv2.CAP_PROP_FPS) or 24.0
probe_cap.release()
print(f"fps={FPS:.2f}  extracting {len(shots)} keyframes")

def grab_cv2(cap, sec):
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000.0)
    ok, frame = cap.read()
    if ok and frame is not None:
        return frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * FPS))
    ok, frame = cap.read()
    return frame if (ok and frame is not None) else None

def grab_ffmpeg(sec, dest):
    r = subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{sec:.3f}",
         "-i", str(CLIP_PATH), "-frames:v", "1", "-q:v", "2", str(dest)],
        capture_output=True, text=True)
    return r.returncode == 0 and dest.exists() and dest.stat().st_size > 1000

t0 = time.time()
cap = cv2.VideoCapture(str(CLIP_PATH))
if not cap.isOpened():
    raise RuntimeError(f"opencv cannot open {CLIP_PATH}")

failed = []
for s in shots:
    dest = FRAMES / f"shot_{s.idx:05d}.jpg"
    frame = grab_cv2(cap, s.mid)
    if frame is None:
        frame = grab_cv2(cap, s.start + 0.1)
    if frame is not None:
        cv2.imwrite(str(dest), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        s.frame_path = str(dest)
    elif grab_ffmpeg(s.mid, dest):
        s.frame_path = str(dest)
    else:
        failed.append(s.idx)
cap.release()

kept = [s for s in shots if s.frame_path and Path(s.frame_path).exists()]
if not kept:
    raise RuntimeError(
        f"extracted 0 keyframes from {CLIP_PATH}\n"
        "opencv and ffmpeg both failed to read frames - the clip may be corrupt.")

if failed:
    print(f"dropped {len(failed)} shots with unreadable frames: {failed[:10]}")

shots = kept
for i, s in enumerate(shots):
    s.idx = i

TIMINGS["keyframes"] = time.time() - t0
print(f"{len(shots)} keyframes in {TIMINGS['keyframes']:.0f}s")

In [ ]:
usable = [s for s in shots if s.frame_path and Path(s.frame_path).exists()]
if not usable:
    raise RuntimeError("no keyframes on disk - run the keyframe cell above")

step = max(1, len(usable) // 18)
picks = usable[::step][:18]

fig, axes = plt.subplots(3, 6, figsize=(18, 7))
for ax, s in zip(axes.ravel(), picks):
    ax.imshow(Image.open(s.frame_path)); ax.axis("off")
    ax.set_title(f"#{s.idx}  {int(s.start//60)}:{int(s.start%60):02d}", fontsize=8)
for ax in axes.ravel()[len(picks):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

---
## 3. Visual pass — CLIP

CLIP puts images and text into **one shared vector space**, which is what lets a typed query be compared against a picture at all.

Know its limits going in: CLIP is strong on objects, places and mood, and weak on *actions and relations between people*. It will find "a rooftop at night" easily and struggle with "ties another man upside down". That weakness is the entire reason the caption pass exists — and the ablation in section 9 will show you exactly how large it is.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

t0 = time.time()
clip_model = CLIPModel.from_pretrained(CLIP_MODEL, torch_dtype=DTYPE).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_MODEL)

EMBED_DIM = clip_model.config.projection_dim

def pooled_from(out):
    pooled = getattr(out, "pooler_output", None)
    if pooled is None:
        pooled = out.last_hidden_state[:, 0]
    return pooled

def embed_images(paths, batch=32):
    out = []
    for i in range(0, len(paths), batch):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i+batch]]
        inp = clip_proc(images=imgs, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            vout = clip_model.vision_model(pixel_values=inp["pixel_values"].to(DTYPE))
            f = clip_model.visual_projection(pooled_from(vout))
        out.append(torch.nn.functional.normalize(f, dim=-1).float().cpu().numpy())
    return np.vstack(out)

def embed_clip_text(texts):
    inp = clip_proc(text=texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        tout = clip_model.text_model(input_ids=inp["input_ids"], attention_mask=inp.get("attention_mask"))
        f = clip_model.text_projection(pooled_from(tout))
    return torch.nn.functional.normalize(f, dim=-1).float().cpu().numpy()

visual_emb = embed_images([s.frame_path for s in shots])

if visual_emb.shape[1] != EMBED_DIM:
    raise RuntimeError(
        f"expected {EMBED_DIM}-d embeddings, got {visual_emb.shape[1]}. "
        "CLIP's projection layer was not applied - image and text vectors will not be comparable.")

probe = embed_clip_text(["a man in a metal suit flying through the sky",
                         "a bowl of spaghetti on a wooden table"])
if probe.shape[1] != visual_emb.shape[1]:
    raise RuntimeError(f"image dim {visual_emb.shape[1]} != text dim {probe.shape[1]}")

hit, miss = (visual_emb @ probe[0]).max(), (visual_emb @ probe[1]).max()
print(f"sanity: relevant query peaks at {hit:.3f}, irrelevant at {miss:.3f}")
if hit < 0.20:
    raise RuntimeError(
        f"peak image-text cosine {hit:.3f} is far below the ~0.25-0.35 CLIP normally gives.\n"
        "Image and text are not landing in the same space - check the projection wiring.")
if hit - miss < 0.03:
    raise RuntimeError(f"relevant ({hit:.3f}) and irrelevant ({miss:.3f}) queries score the same")

TIMINGS["clip_visual"] = time.time() - t0
print(f"visual embeddings {visual_emb.shape} in {TIMINGS['clip_visual']:.0f}s "
      f"({TIMINGS['clip_visual']/len(shots)*1000:.0f} ms/shot)")
print(f"image and text both {EMBED_DIM}-d - shared space confirmed")

---
## 4. Dialogue pass — Whisper

Word-level timestamps, attached to whichever shots they overlap.

Expect this signal to be **empty exactly when you need it most**: action sequences have no dialogue. That is not a flaw, it is the argument for fusing multiple signals — they fail in different places.

In [ ]:
from faster_whisper import WhisperModel

t0 = time.time()
asr = WhisperModel(WHISPER_MODEL, device=DEVICE, compute_type="float16" if DEVICE=="cuda" else "int8")
segments, info = asr.transcribe(str(AUDIO_PATH), word_timestamps=True, vad_filter=True)

words = []
for seg in segments:
    if seg.words:
        for w in seg.words:
            words.append((w.start, w.end, w.word))
    else:
        words.append((seg.start, seg.end, seg.text))

for s in shots:
    s.dialogue = "".join(w for (a, b, w) in words if b > s.start and a < s.end).strip()

TIMINGS["whisper"] = time.time() - t0
spoken = sum(1 for s in shots if s.dialogue)
print(f"{len(words)} words, language={info.language}, in {TIMINGS['whisper']:.0f}s")
print(f"{spoken}/{len(shots)} shots have dialogue ({spoken/len(shots)*100:.0f}%)")

---
## 5. Caption pass — VLM

This is the pass that makes **action queries** work, and it is by far the most expensive one. A vision-language model looks at each keyframe and writes what is happening.

The prompt is deliberately action-focused. Ask a VLM to "describe this image" and it returns *"a man standing in a room"* — a fluent sentence carrying no retrievable information. Ask what people are **doing to each other**, and you get the verbs and relations that queries are actually made of.

Whether this pass earns its cost is a real open question, and section 9 answers it with numbers rather than opinion.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

t0 = time.time()
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL, torch_dtype=DTYPE, device_map="auto").eval()
vlm_proc = AutoProcessor.from_pretrained(VLM_MODEL, min_pixels=256*28*28, max_pixels=768*28*28)

CAPTION_PROMPT = (
    "Describe the physical action in one short sentence. "
    "Begin directly with the subject performing the action. "
    "Name what each person is doing to other people, vehicles or objects. "
    "Use concrete verbs. "
    "Never begin with 'In the frame', 'In this image' or 'The image shows'. "
    "Do not describe mood, style, lighting or camera work."
)

PREAMBLE = re.compile(r"^(in|from)?\s*(this|the)?\s*(frame|image|picture|photo|scene|shot)[,:]?\s+", re.I)
HEDGE = re.compile(r"\b(seemingly|apparently|appears to be|seems to be|possibly|likely|presumably)\s*", re.I)
FILLER = re.compile(r"\b(a superhero|the superhero|a character|the character)\b[,]?\s*", re.I)

def clean_caption(t):
    t = t.strip().strip('"')
    for _ in range(2):
        t = PREAMBLE.sub("", t).strip()
    t = HEDGE.sub("", t)
    t = FILLER.sub("", t)
    t = re.sub(r"\s{2,}", " ", t).strip(" ,")
    return t[:1].upper() + t[1:] if t else t

def caption(path):
    msg = [{"role":"user","content":[{"type":"image","image":path},{"type":"text","text":CAPTION_PROMPT}]}]
    text = vlm_proc.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    img = Image.open(path).convert("RGB")
    inp = vlm_proc(text=[text], images=[img], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = vlm.generate(**inp, max_new_tokens=60, do_sample=False)
    trimmed = out[0][len(inp.input_ids[0]):]
    return vlm_proc.decode(trimmed, skip_special_tokens=True).strip()

for n, s in enumerate(shots):
    s.caption = clean_caption(caption(s.frame_path))
    if n % 25 == 0:
        print(f"  {n}/{len(shots)}  {s.caption[:70]}")

TIMINGS["captions"] = time.time() - t0
print(f"\ncaptioned {len(shots)} shots in {TIMINGS['captions']:.0f}s "
      f"({TIMINGS['captions']/len(shots):.1f} s/shot)")

In [ ]:
for s in shots[:8]:
    print(f"#{s.idx:3d} {int(s.start//60)}:{int(s.start%60):02d}  {s.caption}")
    if s.dialogue:
        print(f"           dialogue: {s.dialogue[:80]}")

---
## 6. Shots into scenes — semantic chunking

Here is the decision that separates a demo that feels magic from one that feels broken, at **identical retrieval scores**.

A shot is one camera take. A *scene* is one dramatic unit, and it is usually a dozen shots — the editor cuts between faces and angles while a single continuous event unfolds. "The man ties the other man upside down" is not a shot; it is fifteen shots over ninety seconds.

So we use **parent-child**: embed shots (small, precise, one visual idea each), but return scenes (enough context to be useful, and the right place to seek). If we returned the matching shot instead, the player would drop the viewer into the rope close-up eleven seconds in, with no idea how they got there.

The merge rule below is **semantic chunking** — the same technique text RAG uses on sentences, applied to shots: keep merging while consecutive embeddings stay similar, cut at a visual discontinuity. `SCENE_MERGE_SIM` is the main quality knob in the entire feature.

In [ ]:
@dataclass
class Scene:
    id: int
    start: float
    end: float
    shot_ids: list = field(default_factory=list)
    @property
    def duration(self): return self.end - self.start

def group_into_scenes(shots, emb, sim_thresh, max_sec, max_gap, min_sec, window=3):
    groups, cur = [], [0]
    for i in range(1, len(shots)):
        recent = emb[cur[-window:]].mean(axis=0)
        recent /= np.linalg.norm(recent) + 1e-9
        sim = float(np.dot(emb[i], recent))
        gap = shots[i].start - shots[i-1].end
        span = shots[i].end - shots[cur[0]].start
        talky = bool(shots[i-1].dialogue.strip() and shots[i].dialogue.strip())
        close_enough = sim >= sim_thresh or (talky and sim >= sim_thresh - 0.15)
        if close_enough and gap <= max_gap and span <= max_sec:
            cur.append(i)
        else:
            groups.append(cur)
            cur = [i]
    groups.append(cur)

    absorbed = []
    for ids in groups:
        dur = shots[ids[-1]].end - shots[ids[0]].start
        if absorbed and dur < min_sec:
            prev = absorbed[-1]
            if shots[ids[-1]].end - shots[prev[0]].start <= max_sec:
                prev.extend(ids)
                continue
        absorbed.append(list(ids))

    out = []
    for sid, ids in enumerate(absorbed):
        sc = Scene(id=sid, start=shots[ids[0]].start, end=shots[ids[-1]].end, shot_ids=ids)
        for i in ids:
            shots[i].scene_id = sid
        out.append(sc)
    return out

scenes = group_into_scenes(shots, visual_emb, SCENE_MERGE_SIM, SCENE_MAX_SEC, SCENE_MAX_GAP, SCENE_MIN_SEC, SCENE_WINDOW)

durs = [s.duration for s in scenes]
singles = sum(1 for s in scenes if len(s.shot_ids) == 1)
print(f"{len(shots)} shots -> {len(scenes)} scenes")
print(f"mean {np.mean(durs):.1f}s  median {np.median(durs):.1f}s  max {max(durs):.1f}s")
print(f"mean {len(shots)/len(scenes):.1f} shots/scene, {singles} single-shot scenes ({singles/len(scenes)*100:.0f}%)")
if singles / len(scenes) > 0.25:
    print("\nmany single-shot scenes - lower SCENE_MERGE_SIM to merge more")
at_cap = sum(1 for d in durs if d >= SCENE_MAX_SEC - 0.5)
if at_cap:
    print(f"{at_cap}/{len(scenes)} scenes pinned at the {SCENE_MAX_SEC}s cap - cut by length, not content")

---
## 7. Text embeddings, with context

One more RAG technique worth knowing: **contextual retrieval**.

A caption ripped out of its film loses the context needed to interpret it. *"A man ties another man's ankles"* is much harder to match than *"In a rooftop confrontation between two men: a man ties another man's ankles."* So before embedding, we prepend a short description of where the shot sits.

It is cheap, and it measurably reduces retrieval misses.

Each shot ends up with **three vectors** — visual, caption, dialogue. Three signals that fail in different places, which is exactly why we keep all three.

In [ ]:
from sentence_transformers import SentenceTransformer

t0 = time.time()
text_model = SentenceTransformer(TEXT_MODEL, device=DEVICE)

def scene_context(sc):
    caps = [shots[i].caption for i in sc.shot_ids][:3]
    return " ".join(caps)[:200]

ctx_by_scene = {sc.id: scene_context(sc) for sc in scenes}

caption_texts, dialogue_texts, dialogue_owner = [], [], []
for s in shots:
    caption_texts.append(f"Scene context: {ctx_by_scene[s.scene_id]} | Moment: {s.caption}")
    if s.dialogue:
        dialogue_texts.append(s.dialogue)
        dialogue_owner.append(s.idx)

caption_emb = text_model.encode(caption_texts, normalize_embeddings=True,
                                batch_size=64, show_progress_bar=False)
dialogue_emb = (text_model.encode(dialogue_texts, normalize_embeddings=True,
                                  batch_size=64, show_progress_bar=False)
                if dialogue_texts else np.zeros((0, caption_emb.shape[1])))

def embed_query_text(q):
    return text_model.encode([f"Represent this sentence for searching relevant passages: {q}"],
                             normalize_embeddings=True)[0]

TIMINGS["text_embed"] = time.time() - t0
print(f"caption {caption_emb.shape}  dialogue {dialogue_emb.shape}  in {TIMINGS['text_embed']:.0f}s")

---
## 8. Retrieval and fusion

Three independent rankings, combined with **Reciprocal Rank Fusion**:

```
score(item) = sum over rankings of  1 / (k + rank)
```

RRF only looks at **rank position**, never at raw scores. That is precisely why it is the right choice here: a CLIP cosine and a BGE cosine are not on comparable scales, and any attempt to normalise or weight them is a tuning exercise with no principled answer. RRF sidesteps the problem entirely and has no parameters worth arguing about.

Shot-level hits are then collapsed to their parent scene, keeping the best rank per scene. Retrieval is exact numpy cosine here — a few hundred shots do not need an index, and exact search means the numbers below measure *retrieval quality* rather than ANN approximation error. Chroma slots in behind the `VectorStore` seam later.

In [ ]:
def rank_visual(q):
    qv = embed_clip_text([q])[0]
    sims = visual_emb @ qv
    return np.argsort(-sims), sims

def rank_caption(q):
    qv = embed_query_text(q)
    sims = caption_emb @ qv
    return np.argsort(-sims), sims

def rank_dialogue(q):
    if len(dialogue_emb) == 0:
        return np.array([], dtype=int), np.array([])
    qv = embed_query_text(q)
    sims = dialogue_emb @ qv
    order = np.argsort(-sims)
    return np.array([dialogue_owner[i] for i in order]), sims

def standout(sims):
    if len(sims) < 3:
        return 0.0
    return float((sims.max() - sims.mean()) / (sims.std() + 1e-9))

def to_scene_ranking(shot_order, limit=None):
    if limit is None:
        limit = max(3, len(scenes) // 3)
    seen, out, best_shot = set(), [], {}
    for si in shot_order:
        si = int(si)
        sc = shots[si].scene_id
        if sc not in seen:
            seen.add(sc); out.append(sc); best_shot[sc] = si
        if limit and len(out) >= limit:
            break
    return out, best_shot

def search(q, variant="fused", top_k=TOP_K, depth=FUSE_DEPTH, z=SIGNAL_Z, explain=False):
    wanted = []
    if variant in ("visual", "fused", "visual+caption"):
        wanted.append(("visual", rank_visual))
    if variant in ("caption", "fused", "visual+caption"):
        wanted.append(("caption", rank_caption))
    if variant in ("dialogue", "fused"):
        wanted.append(("dialogue", rank_dialogue))

    rankings, notes, evidence = [], [], {}
    for name, fn in wanted:
        order, sims = fn(q)
        if len(order) == 0:
            notes.append(f"{name}: empty")
            continue
        conf = standout(sims)
        peak = float(sims.max())
        solo = len(wanted) == 1
        if conf < z and not solo:
            notes.append(f"{name}: abstained (z={conf:.2f} max={peak:.3f})")
            continue
        notes.append(f"{name}: voted (z={conf:.2f} max={peak:.3f})")
        ranking, best = to_scene_ranking(order, depth)
        rankings.append(ranking)
        for sid, shot_i in best.items():
            evidence.setdefault(sid, []).append((name, shot_i))

    fused = {}
    for ranking in rankings:
        for rank, sid in enumerate(ranking):
            fused[sid] = fused.get(sid, 0.0) + 1.0 / (RRF_K + rank + 1)
    order = sorted(fused, key=fused.get, reverse=True)[:top_k]
    result = [(scenes[sid], fused[sid], evidence.get(sid, [])) for sid in order]
    return (result, notes) if explain else result

def stamp(t):
    return f"{int(t//60)}:{int(t%60):02d}"

def show(q, variant="fused", n=5):
    result, notes = search(q, variant, explain=True)
    print(f"query: {q}   [{variant}]")
    print(f"  signals: {'; '.join(notes)}")
    for rank, (sc, score, ev) in enumerate(result[:n], 1):
        print(f"  {rank}. {stamp(sc.start)}-{stamp(sc.end)}  ({score:.4f})  {len(sc.shot_ids)} shots")
        for name, shot_i in ev:
            sh = shots[shot_i]
            detail = sh.caption if name != "dialogue" else (sh.dialogue or sh.caption)
            print(f"       via {name:<8} @{stamp(sh.start)}  {detail[:78]}")

show("two fighter jets chasing iron man through the sky")

In [ ]:
PROBE_QUERIES = {
    "visual_action":   ["two fighter jets chasing iron man through the sky",
                        "iron man catches the falling pilot in midair",
                        "a jet explodes and falls out of the sky"],
    "spoken":          ["what is going on out there",
                        "we have an unidentified bogey",
                        "stand down"],
    "nonsense":        ["a giraffe eating a birthday cake in a swimming pool",
                        "medieval knights jousting on horseback",
                        "a chef chopping vegetables in a kitchen"],
}

print(f"{'kind':<14}{'query':<44}{'visual':>9}{'caption':>9}{'dialogue':>10}")
print("-" * 86)
for kind, qs in PROBE_QUERIES.items():
    for q in qs:
        _, sv = rank_visual(q)
        _, sc = rank_caption(q)
        _, sd = rank_dialogue(q)
        print(f"{kind:<14}{q[:42]:<44}{sv.max():>9.3f}{sc.max():>9.3f}{(sd.max() if len(sd) else float('nan')):>10.3f}")
print("-" * 86)
print("relevant queries should peak clearly above the nonsense rows")
print("set each signal's floor between the two")

---
## 9. Ground truth — the part that must be yours

**Without labels this notebook proves nothing.** You will scroll the results, they will look plausible, and you will conclude it works. That is not evidence.

So: run the contact sheet below, scrub the clip, and write down **~15 queries with the timestamp where each really begins.** Phrase them the way a user would — natural language, not keywords lifted from the captions. Include hard cases: pure action with no dialogue, dialogue-only exchanges, and something visually similar to another scene.

This is thirty minutes of work and it is what converts every later decision from an argument into a measurement.

In [ ]:
fig, axes = plt.subplots(max(1, math.ceil(len(scenes)/6)), 6,
                         figsize=(18, 2.6*max(1, math.ceil(len(scenes)/6))))
axes = np.atleast_2d(axes)
for ax, sc in zip(axes.ravel(), scenes):
    ax.imshow(Image.open(shots[sc.shot_ids[0]].frame_path)); ax.axis("off")
    ax.set_title(f"scene {sc.id} @ {int(sc.start//60)}:{int(sc.start%60):02d} ({sc.duration:.0f}s)", fontsize=8)
for ax in axes.ravel()[len(scenes):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
GROUND_TRUTH = [
    {"q": "iron man flies up alongside the two fighter jets", "t": None},
    {"q": "a jet clips iron man and the plane spins out of control", "t": None},
    {"q": "the pilot ejects from the damaged jet", "t": None},
    {"q": "the pilot's parachute is jammed and will not open", "t": None},
    {"q": "iron man catches the falling pilot in midair", "t": None},
    {"q": "the parachute finally opens and the pilot floats down", "t": None},
    {"q": "iron man clings to the top of a jet", "t": None},
    {"q": "the jets fire missiles at him", "t": None},
    {"q": "he dodges a missile by climbing straight up", "t": None},
    {"q": "rhodey is on the phone asking what is going on", "t": None},
    {"q": "someone reports an unidentified bogey on the radio", "t": None},
    {"q": "rhodey tells them to stand down", "t": None},
    {"q": "iron man flying alone over the clouds", "t": None},
    {"q": "a jet banking hard through open sky", "t": None},
    {"q": "tony lands back at his house", "t": None},
]

missing = [g["q"] for g in GROUND_TRUTH if g["t"] is None]
if missing:
    print(f"{len(missing)} of {len(GROUND_TRUTH)} queries still need a timestamp\n")
    for q in missing:
        print(f'    t=None    {q}')
    print("\nScrub the contact sheet above, fill every t, drop any query whose scene")
    print("is not in your clip, then re-run. Section 10 stays blocked until this is done.")
else:
    print(f"{len(GROUND_TRUTH)} labelled queries ready")

---
## 10. Evaluation and ablation

A **hit** means the returned scene starts within `HIT_TOLERANCE_SEC` of the truth.

The ablation is the genuinely valuable output. It tells you which passes earn their cost, and it sizes everything downstream:

- If **CLIP alone** gets most of the way, the expensive VLM pass dies and the CPU problem largely evaporates.
- If **captions** carry it, borrowed GPU is not optional and the `Embedder` seam becomes load-bearing.
- If **fused** does not beat its best single component, fusion is adding noise and something is misconfigured.

In [ ]:
def evaluate(variant, tol=HIT_TOLERANCE_SEC):
    r1 = r5 = 0; rr = []; errs = []
    for item in GROUND_TRUTH:
        results = search(item["q"], variant, top_k=10)
        hit_rank = None
        for rank, (sc, *_ ) in enumerate(results, 1):
            if abs(sc.start - item["t"]) <= tol or (sc.start - tol <= item["t"] <= sc.end):
                hit_rank = rank; break
        if hit_rank:
            if hit_rank == 1: r1 += 1
            if hit_rank <= 5: r5 += 1
            rr.append(1.0 / hit_rank)
        else:
            rr.append(0.0)
        if results:
            errs.append(abs(results[0][0].start - item["t"]))
    n = len(GROUND_TRUTH)
    return {"recall@1": r1/n, "recall@5": r5/n, "mrr": float(np.mean(rr)),
            "median_err_s": float(np.median(errs)) if errs else float("nan")}

if GROUND_TRUTH and not any(g["t"] is None for g in GROUND_TRUTH):
    variants = ["dialogue", "visual", "caption", "visual+caption", "fused"]
    rows = {v: evaluate(v) for v in variants}
    print(f"{'variant':<16}{'R@1':>8}{'R@5':>8}{'MRR':>8}{'med err':>10}")
    print("-" * 50)
    for v, m in rows.items():
        print(f"{v:<16}{m['recall@1']:>8.2f}{m['recall@5']:>8.2f}{m['mrr']:>8.3f}{m['median_err_s']:>9.1f}s")
    best_single = max((m["mrr"], v) for v, m in rows.items() if v != "fused")
    print(f"\nbest single signal: {best_single[1]} (MRR {best_single[0]:.3f})")
    print(f"fusion gain: {rows['fused']['mrr'] - best_single[0]:+.3f} MRR")

### Where the time went

These numbers size the real pipeline. Scale them up: a 2-hour film is roughly 12x a 10-minute clip, and this ran on a T4 — the laptop has no CUDA at all, so divide the GPU throughput by a large factor when planning what runs locally.

Whichever stage dominates here is the one the `Embedder` seam exists for.

In [ ]:
total = sum(TIMINGS.values())
print(f"{'stage':<20}{'seconds':>10}{'share':>9}{'per 2h film':>14}")
print("-" * 54)
for k, v in sorted(TIMINGS.items(), key=lambda kv: -kv[1]):
    scaled = v * (7200 / CLIP_SECONDS)
    print(f"{k:<20}{v:>10.0f}{v/total*100:>8.0f}%{scaled/60:>12.0f}m")
print("-" * 54)
print(f"{'TOTAL':<20}{total:>10.0f}{100:>8.0f}%{total*(7200/CLIP_SECONDS)/60:>12.0f}m")

---
## 11. Try it yourself

Type queries the way a viewer would. Compare `fused` against the single signals on the same query — that is where the intuition for what each one actually contributes comes from.

In [ ]:
for v in ["visual", "caption", "dialogue", "fused"]:
    show("a man ties up another man", variant=v, n=3)
    print()

In [ ]:
show("iron man catches the falling pilot in midair", variant="fused", n=5)

---
## What to do with the result

**If it works** — carry forward the ablation table, the chosen models, and the per-stage timings. Those three things size the real pipeline: which passes to keep, what the CPU budget must be, and whether the GPU escape hatch is needed from day one.

**If it does not** — a day is spent, not a month, and there are levers left before abandoning the idea:

- larger CLIP, or a video-native model instead of frame-based
- a stronger VLM, or captioning several frames per shot instead of one
- tune `SHOT_THRESHOLD` and `SCENE_MERGE_SIM` against the labels — mis-set, these alone can sink recall
- query rewriting: expand the user's phrasing before embedding
- check the failures by hand. "Retrieval is bad" and "scene boundaries are wrong" look identical in the metrics and have completely different fixes

Either way the next step is the same: **read the failures, not just the averages.**